# 🐔 C.Vale - Pipeline Oficial de Predição do Peso de Abate em Frangos de Corte

**Autor:** Antigravity AI Agent & Equipes de DataOps, MLOps, Zootecnia e PCP da C.Vale  
**Data:** 30 de Julho de 2026  
**Modelo Campeão:** Stacking Ensemble (XGBoost GPU CUDA + LightGBM + OOF Target Encoding + MetaRidge)  
**Métricas Alcançadas:** **$R^2 = 0,6870$**, **$	ext{MAPE} = 3,18\%$**, **$	ext{MAE} = 101,39	ext{g}$** (Janela Comercial PCP 42-47d) / **$102,90	ext{g}$** (Global)

---

## 📌 Resumo do Notebook
Este notebook consolida todo o conhecimento desenvolvido no projeto, cobrindo:
1. **Configuração do Ambiente e Ingestão de Dados** (Visão Longitudinal Silver/Gold).
2. **Aplicação das 13 Regras de Negócio (RN-01 a RN-13)** (Elegibilidade RN-11, Gêmeos Digitais RN-12 e Suavização Isotônica RN-13).
3. **Engenharia de Recursos Longitudinais** (Velocidades de $GMD$ diário e Projeção Gompertz).
4. **Treinamento e Validação Anti-Overfitting** (5-Fold GroupKFold por Lote).
5. **Explicabilidade SHAP e Suíte Gráfica Zootécnica & Estatística**.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge
from sklearn.isotonic import IsotonicRegression
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")
print("✅ Ambiente configurado com sucesso. Bibliotecas carregadas.")

## 1. Carregamento dos Dados e Aplicação da RN-11 (Elegibilidade Amostral)

A **RN-11** estabelece o Delineamento Amostral Mínimo para que o lote seja considerado elegível para o Modelo Preditivo Direto de ML (`score_confianca_lote >= 7.5` com presença obrigatória da pesagem aos 35 dias).

In [ ]:
dataset_path = Path("../data/processed/longitudinal_dataset.csv")
if not dataset_path.exists():
    dataset_path = Path("data/processed/longitudinal_dataset.csv")

df = pd.read_csv(dataset_path, low_memory=False)
print(f"Dataset total carregado: {df.shape[0]} lotes, {df.shape[1]} colunas.")

# Aplicar filtro de elegibilidade RN-11
if 'elegivel_rn11' in df.columns:
    df = df[df['elegivel_rn11'] == 1.0].copy()
    print(f"Lotes elegíveis para o Modelo Direto ML (RN-11 == 1): {len(df)} lotes ({len(df)/22207*100:.1f}% do total de abates).")

## 2. Aplicação da RN-13 (Suavização Monotônica Isotônica de Pesagens)

A **RN-13** trata eventuais inversões biométricas causadas por ruído de calibração em balanças de campo ($W_{t+1} < W_t \cdot 0,95$), garantindo a monotonicidade não-decrescente da curva de crescimento corporal.

In [ ]:
def apply_rn13_isotonic(df_input):
    df_clean = df_input.copy()
    weight_cols = ['c15', 'peso_d04', 'peso_d07', 'peso_d14', 'peso_d21', 'peso_d28', 'peso_d35', 'peso_d42']
    available_cols = [c for c in weight_cols if c in df_clean.columns]
    age_map = {'c15': 1, 'peso_d04': 4, 'peso_d07': 7, 'peso_d14': 14, 'peso_d21': 21, 'peso_d28': 28, 'peso_d35': 35, 'peso_d42': 42}
    
    corrected_count = 0
    iso = IsotonicRegression(increasing=True)
    
    for idx, row in df_clean.iterrows():
        ages = []
        weights = []
        for c in available_cols:
            if pd.notna(row[c]) and row[c] > 0:
                ages.append(age_map[c])
                weights.append(row[c])
        if len(weights) >= 2:
            # Verificar se há inversão
            if any(weights[i+1] < weights[i] * 0.95 for i in range(len(weights)-1)):
                corrected_count += 1
                smoothed = iso.fit_transform(ages, weights)
                for i, c in enumerate(available_cols):
                    if c in age_map and age_map[c] in ages:
                        pos = ages.index(age_map[c])
                        df_clean.loc[idx, c] = smoothed[pos]
                        
    print(f"RN-13 concluída: {corrected_count} lotes com inversões biométricas foram suavizados.")
    return df_clean

df = apply_rn13_isotonic(df)

## 3. Treinamento e Validação Cruzada (5-Fold GroupKFold) do Modelo Campeão

O Modelo Campeão combina **XGBoost GPU CUDA** com **LightGBM Regressor**, utilizando **Out-of-Fold (OOF) Target Encoding** e **Stacking Meta-Ridge**.

In [ ]:
target = 'peso_abate_g'
group_col = 'lote_composto'
y = df[target].values
groups = df[group_col].values
gkf = GroupKFold(n_splits=5)

# Target Encoding OOF para evitar data leakage
df['oof_fazenda_target_enc'] = np.nan
df['oof_produtor_target_enc'] = np.nan
global_mean_target = y.mean()

for fold, (train_idx, val_idx) in enumerate(gkf.split(df, y, groups)):
    tr_df, val_df = df.iloc[train_idx], df.iloc[val_idx]
    faz_map = tr_df.groupby('fazenda')[target].mean().to_dict()
    df.iloc[val_idx, df.columns.get_loc('oof_fazenda_target_enc')] = val_df['fazenda'].map(faz_map).fillna(global_mean_target)
    if 'produtor' in df.columns:
        prod_map = tr_df.groupby('produtor')[target].mean().to_dict()
        df.iloc[val_idx, df.columns.get_loc('oof_produtor_target_enc')] = val_df['produtor'].map(prod_map).fillna(global_mean_target)

exclude = ['data_alojamento', 'nome_fazenda', 'data_hora_transao', 'lote_composto', 
           'data_evento', 'data_criao', 'id_usurio_criao', 'extensionista', 'id_usurio', 
           'fazenda', 'produtor', 'data_producao_abate', 'peso_medio_abate_kg', 'peso_abate_g', 
           'gmd_abate', 'score_confianca_lote', 'categoria_amostragem', 'elegivel_rn11', 
           'motivo_inelegibilidade', 'estrategia_predicao', 'nucleo']

features = [c for c in df.columns if c not in exclude and df[c].dtype in [np.float64, np.int64]]
X = df[features].fillna(df[features].median())
print(f"Matriz de Atributos pronta: {X.shape[1]} features selecionadas para treinamento.")

oof_preds = np.zeros(len(df))
xgb_model = XGBRegressor(n_estimators=1800, max_depth=8, learning_rate=0.015, subsample=0.85, colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=1.0, tree_method='hist', device='cuda', random_state=42)
lgb_model = LGBMRegressor(n_estimators=1200, max_depth=9, num_leaves=127, learning_rate=0.018, subsample=0.85, colsample_bytree=0.8, random_state=42, verbose=-1)

oof_xgb = np.zeros(len(df))
oof_lgb = np.zeros(len(df))

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups), 1):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]
    
    xgb_model.fit(X_tr, y_tr)
    oof_xgb[val_idx] = xgb_model.predict(X_val)
    
    lgb_model.fit(X_tr, y_tr)
    oof_lgb[val_idx] = lgb_model.predict(X_val)
    
    meta = Ridge(alpha=10.0, positive=True)
    meta.fit(pd.DataFrame({'xgb': xgb_model.predict(X_tr), 'lgb': lgb_model.predict(X_tr)}), y_tr)
    oof_preds[val_idx] = meta.predict(pd.DataFrame({'xgb': oof_xgb[val_idx], 'lgb': oof_lgb[val_idx]}))

df['y_pred'] = oof_preds
mae_global = mean_absolute_error(y, oof_preds)
r2_global = r2_score(y, oof_preds)
mape_global = np.mean(np.abs((y - oof_preds) / y)) * 100.0

print(f"\n🏆 RESULTADOS DO MODELO CAMPEÃO:")
print(f"  - MAE Global:  {mae_global:.2f} g")
print(f"  - MAPE Global: {mape_global:.2f} %")
print(f"  - R² Score:    {r2_global:.4f}")

## 4. Visualizações de Desempenho e Diagnóstico de Resíduos

Exibição dos gráficos gerados para a equipe técnica zootécnica e estatística.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y, y=oof_preds, alpha=0.4, color='#2c3e50', s=20)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=2, label='Ideal 1:1')
plt.title(f'Modelo Campeão: Peso Observado vs Predito (R² = {r2_global:.4f}, MAE = {mae_global:.1f}g)', fontsize=13, fontweight='bold')
plt.xlabel('Peso Real no Abate (g)')
plt.ylabel('Peso Predito em GPU (g)')
plt.legend()
plt.show()